# Lakeshore inventory semantic lifecycle
Run each cell separately. Change WORKSPACE and MODEL to your own names. Inspect existing relationships before the write cell; do not create parallel relationships.


In [ ]:
import sempy.fabric as fabric
WORKSPACE = "LakeshoreProgressionDev"
MODEL = "LakeshoreInventorySemanticDev"
display(fabric.list_tables(MODEL, workspace=WORKSPACE))
display(fabric.list_columns(MODEL, workspace=WORKSPACE))


In [ ]:
fact = fabric.read_table(MODEL, "FactInventorySnapshot", workspace=WORKSPACE)
product = fabric.read_table(MODEL, "DimProductModel", workspace=WORKSPACE)
store = fabric.read_table(MODEL, "DimStoreModel", workspace=WORKSPACE)
dates = fabric.read_table(MODEL, "DimDateModel", workspace=WORKSPACE)
print("Rows", len(fact), "Units", fact.OnHandQuantity.sum())
print("Duplicate grain", fact.duplicated(["StoreKey", "ProductKey", "DateKey"]).sum())
for key, dim in [("ProductKey",product),("StoreKey",store),("DateKey",dates)]:
    print(key, "null", fact[key].isna().sum(), "orphans", (~fact[key].isin(dim[key])).sum())


In [ ]:
from sempy.relationships import find_relationships, list_relationship_violations
frames = [fact, product, store, dates]
relationships = find_relationships(frames)
display(relationships)
display(list_relationship_violations(frames, relationships))


In [ ]:
dax = """EVALUATE SUMMARIZECOLUMNS('DimProductModel'[StorageClass], "Positions", COUNTROWS('FactInventorySnapshot'), "OnHandUnits", SUM('FactInventorySnapshot'[OnHandQuantity])) ORDER BY 'DimProductModel'[StorageClass]"""
display(fabric.evaluate_dax(MODEL, dax, workspace=WORKSPACE))


In [ ]:
import Microsoft.AnalysisServices.Tabular as TOM
with fabric.connect_semantic_model(MODEL, workspace=WORKSPACE, readonly=False) as tom:
    for key, target in [("ProductKey","DimProductModel"),("StoreKey","DimStoreModel"),("DateKey","DimDateModel")]:
        name = "Inventory_to_" + target
        if tom.model.Relationships.Find(name) is not None:
            print("Already present", name)
            continue
        rel = TOM.SingleColumnRelationship()
        rel.Name = name
        rel.FromColumn = tom.model.Tables["FactInventorySnapshot"].Columns[key]
        rel.ToColumn = tom.model.Tables[target].Columns[key]
        rel.FromCardinality = TOM.RelationshipEndCardinality.Many
        rel.ToCardinality = TOM.RelationshipEndCardinality.One
        rel.CrossFilteringBehavior = TOM.CrossFilteringBehavior.OneDirection
        rel.IsActive = True
        tom.model.Relationships.Add(rel)
print("Three intended relationships configured")


In [ ]:
fabric.refresh_dataset(MODEL, workspace=WORKSPACE)
display(fabric.evaluate_dax(MODEL, dax, workspace=WORKSPACE))
